# 0️⃣5️⃣ - How to parse command outputs with Genie
![Persona](https://img.shields.io/badge/%F0%9F%91%A4%20Persona-%F0%9F%A4%96%20Network%20Automation%20Developer-lightgrey) ![Difficulty](https://img.shields.io/badge/%F0%9F%9A%A6%20Difficulty-Intermediate-yellow) ![RADKit version](https://img.shields.io/badge/RADKit-1.9.6-blue?logo=cisco&logoColor=white) ![Python version](https://img.shields.io/badge/Python-3.12%2B-yellow?logo=python&logoColor=white)

> In this playbook we will explore how to parse device command outputs with Genie parsers.

### 🛠️ Before You Begin

If you haven't done it yet, follow the instructions [in this file](../SETUP.md) to set up your environment and run this playbook.
For the best experience, complete [01-authentication-connection.ipynb](./01-authentication-connection.ipynb) first.

---

### 📋 What you'll learn

| # | Topic |
|---|---|
| 1 | Parse a single command output with `radkit_genie.parse_text()` |
| 2 | Parse multi-device, multi-command responses with `radkit_genie.parse()` |
| 3 | Handle unsupported parsers gracefully with `ParserNotFound` |

---

### 🤖 The `Genie` parser object

`radkit_genie` integrates Genie parsers into your RADKit workflow so you can transform raw CLI text into structured data (`QDict`) and use it directly in your automation logic.

> Genie within RADKit can do **[so much more than parsing](https://radkit.cisco.com/docs/integrations/feature_genie.html)**, but in this notebook we will focus on its parsing functionalities exclusively.

## 🧪 Method 1: Parse a single command output

**Best for:** quick extraction of structured values from one command on one device.

**How it works:**
1. Execute a command with `exec()`.
2. Parse raw output with `radkit_genie.parse_text(...)`.
3. Convert the returned `QDict` to a regular dictionary for printing or downstream processing.

**What you need:**
- A connected RADKit service
- A target device in inventory
- A Genie parser available for the command and platform

---

### 1.1 Why parse raw CLI output?

Raw CLI output is useful for humans, but hard to consume in automation workflows. Genie helps you convert that text into structured fields so your code can access exact values without custom regex or TextFSM templates.

The [Genie SDK](https://developer.cisco.com/docs/genie-docs/), part of the [pyATS framework](https://developer.cisco.com/docs/pyats/), provides this transformation capability and integrates cleanly with RADKit.

---

### 1.2 Execute and parse one command

After retrieving command output, use `parse_text()` with the command and platform type. **Supported options are listed in the [Genie parser catalog](https://pubhub.devnetcloud.com/media/genie-feature-browser/docs/#/parsers).**

Since `QDict` behaves like a mapping, you can convert it to a regular dictionary and pretty-print it with `json.dumps()`.

In [7]:
import radkit_genie
import json
from radkit_client import Client

user_id = input("👤 Enter your CCO user id: ")
service_id = input("🛠️ Enter the service id you want to connect to: ")

with Client.create() as client:
    client.sso_login(user_id)
    service = client.service_cloud(service_id).wait()
    
    # Run a command on a specific device
    my_device = service.inventory['ksp-g02-asr9010-02']
    exec_result_raw = my_device.exec("show version").wait()
    print(f"\n📋Raw command output:\n{exec_result_raw.data}\n--------------\n")
    
    # Parse the output using Genie
    exec_result_parsed = radkit_genie.parse_text(exec_result_raw.data, "show version", "iosxr")
    
    # Convert the QDict to a standard dict before pretty-printing it as JSON
    pretty_json = json.dumps(dict(exec_result_parsed), indent=2)
    print(f"\n🧞‍♂️Parsed command output (friendlier with your code):\n\n{pretty_json}\n")


A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.

📋Raw command output:
RP/0/RSP0/CPU0:KSP-G02-ASR9010-02#show version
Wed Apr  1 12:04:44.612 GMT+4
Cisco IOS XR Software, Version 7.10.2
Copyright (c) 2013-2023 by Cisco Systems, Inc.
 
Build Information:
 Built By     : deenayak
 Built On     : Sat Nov 18 06:46:57 PST 2023
 Built Host   : iox-ucs-035
 Workspace    : /auto/srcarchive14/prod/7.10.2/asr9k-x64/ws
 Version      : 7.10.2
 Location     : /opt/cisco/XR/packages/
 Label        : 7.10.2
 
cisco ASR9K () processor
System uptime is 17 weeks 1 day 18 minutes
 
RP/0/RSP0/CPU0:KSP-G02-ASR9010-02#
--------------


🧞‍♂️Parsed command output (friendlier with your code):

{
  "operating_system": "IOSXR",
  "software_version": "7.10.2",
  "built_by": "deenayak",
  "built_on": "Sat Nov 18 06:46:57 PST 2023",
  "built_host": "iox-ucs-035",
  "device_family": "ASR9K",
  "uptime": "17 weeks 1 day 18 minu

## 🧪 Method 2: Parse multiple devices and commands

**Best for:** normalizing larger execution results so you can iterate by device and command with structured parsed data.

**How it works:**
1. Run multiple commands against multiple devices.
2. Parse the combined response with `radkit_genie.parse()`.
3. Iterate through each device/command result and inspect `status` and parsed `data`.

**What you need:**
- A `DeviceDict` containing your target devices
- A command list supported by Genie parsers for each platform

---

### 2.1 Parse batch command responses

You can parse outputs from an `exec()` request that includes multiple devices and commands. The parsed result keeps the familiar RADKit response structure and exposes parsed `QDict` data in each result object.

In [9]:
import json
import radkit_genie
from radkit_client import Client

user_id = input("👤 Enter your CCO user id: ")
service_id = input("🛠️ Enter the service id you want to connect to: ")

with Client.create() as client:
    client.sso_login(user_id)
    service = client.service_cloud(service_id).wait()
    
    # Our two devices of interest in a DeviceDict object
    my_devices = service.inventory.filter('name', 'p0-2e')
    my_devices.add('ksp-g03-c2960s-16')
    
    commands = [
        "show version",
        "show cdp neighbors",
    ]
    multiple_response = my_devices.exec(commands).wait()
    parsed_response = radkit_genie.parse(multiple_response)
    
    for device_name in ['p0-2e', 'ksp-g03-c2960s-16']:
        print(f"\n📱 {device_name}")
        for command in commands:
            result = parsed_response[device_name][command]
            print(f"🧾 {command}")
            print(f"Status: {result.status}")
            print(f"{json.dumps(dict(result.data.items()), indent=2)}\n----------") # We print the items as a list to avoid printing the entire dictionary if it's too large
    


A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.

📱 p0-2e
🧾 show version
Status: ExecStatus.SUCCESS
{
  "version": {
    "xe_version": "17.18.02",
    "version_short": "17.18",
    "platform": "Catalyst L3 Switch",
    "version": "17.18.2",
    "image_id": "CAT9K_IOSXE",
    "label": "RELEASE SOFTWARE (fc3)",
    "os": "IOS-XE",
    "location": "IOSXE",
    "image_type": "production image",
    "copyright_years": "1986-2025",
    "compiled_date": "Fri 19-Dec-25 03:36",
    "compiled_by": "mcpre",
    "rom": "IOS-XE ROMMON",
    "bootldr": "System Bootstrap, Version 17.15.1r, RELEASE SOFTWARE (P)",
    "hostname": "p0-2E",
    "uptime": "4 weeks, 19 hours, 6 minutes",
    "uptime_this_cp": "4 weeks, 19 hours, 8 minutes",
    "returned_to_rom_by": "PnP reset CLI",
    "returned_to_rom_at": "16:03:02 UTC Tue Mar 3 2026",
    "system_restarted_at": "16:05:41 UTC Tue Mar 3 2026",
    "system_image": "

---

## ⚠️ Parser availability and fallback handling

Not every command has a built-in Genie parser for every platform. If parsing fails with `ParserNotFound`, check the [official parser list](https://pubhub.devnetcloud.com/media/genie-feature-browser/docs/#/parsers).

If a parser does not exist for your use case, you can [create a custom parser](https://pubhub.devnetcloud.com/media/pyats-development-guide/docs/writeparser/writeparser.html) and then [use it in RADKit](https://radkit.cisco.com/docs/integrations/genie_local_parsers.html).

---

### 3.1 Handle unsupported commands safely

In [8]:
import radkit_genie
import json
from radkit_client import Client
from genie.libs.parser.utils.common import ParserNotFound

user_id = input("👤 Enter your CCO user id: ")
service_id = input("🛠️ Enter the service id you want to connect to: ")

with Client.create() as client:
    client.sso_login(user_id)
    service = client.service_cloud(service_id).wait()
    
    # Run a command on a specific device
    my_device = service.inventory['ksp-g02-asr9010-02']
    exec_result_raw = my_device.exec("show watchdog").wait()
    print(f"\n📋Raw command output:\n{exec_result_raw.data}\n--------------\n")
    
    # Parse the output using Genie
    try:
        exec_result_parsed = radkit_genie.parse_text(exec_result_raw.data, "show watchdog", "iosxr")
    except ParserNotFound:
        print("❌ Parser not found for the command 'show watchdog' on platform 'iosxr' ❌")
        exec_result_parsed = None
    
    if exec_result_parsed:
        # Convert the QDict to a standard dict before pretty-printing as JSON
        pretty_json = json.dumps(dict(exec_result_parsed), indent=2)
        print(f"\n🧞‍♂️Parsed command output (friendlier with your code):\n\n{pretty_json}\n")
    else:
        print("ℹ️ Parsing skipped because no parser is available for this command/platform combination.")


A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.

📋Raw command output:
RP/0/RSP0/CPU0:KSP-G02-ASR9010-02#show watchdog
Wed Apr  1 12:05:09.598 GMT+4
---- node0_RSP0_CPU0 ----
Memory information:
    Physical Memory	: 35008.11  MB
    Free Memory		: 28087.605 MB
    Memory State	:   Normal
RP/0/RSP0/CPU0:KSP-G02-ASR9010-02#
--------------

❌ Parser not found for the command 'show watchdog' on platform 'iosxr' ❌
ℹ️ Parsing skipped because no parser is available for this command/platform combination.


## 🔀 Which Method Should I Use?

| Criteria | Method 1: `parse_text()` single command | Method 2: `parse()` batch response | Method 3: `ParserNotFound` fallback |
|---|---|---|---|
| Scope | One device, one command | Multiple devices and/or commands | Any unsupported command case |
| Input type | Raw command text + command + platform | RADKit `exec()` response object | Raw command text + command + platform |
| Best for | Fast extraction in simple scripts | Bulk parsing workflows | Production-safe error handling |
| Output shape | Parsed `QDict` for one command | Parsed response retaining device/command hierarchy | Parsed `QDict` or controlled fallback (`None`) |